# Multimodal Product Search Agent

## Workflow Overview

This notebook implements a **multimodal product search system** using CLIP (SigLIP) for encoding both images and text into a shared vector space.

**How it works:**
1. **Ingest**: Each product's image and text description are encoded into 768-dim vectors using SigLIP, averaged, and stored in Qdrant
2. **Query**: A user provides an image, text, or both → encoded the same way → cosine similarity search finds the closest products
3. **Display**: Results are ranked and displayed with scores and metadata

**Tech Stack:**
- `google/siglip-base-patch16-224` — CLIP encoder (local, free)
- Qdrant — in-memory vector database
- No paid APIs required

In [ ]:
# Cell 1 — Install required packages
!pip install transformers torch torchvision qdrant-client python-dotenv pandas Pillow requests

In [ ]:
# Cell 2 — Imports and setup
import os, io, requests, time, warnings, torch
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from transformers import AutoProcessor, AutoModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("All imports successful ✓")

In [ ]:
# Cell 3 — Load environment variables
load_dotenv()
# No API keys required for now — all encoding is local via SigLIP
# .env is kept as placeholder for future credentials (e.g. Gemini query expansion)
print("Environment loaded ✓ (no API keys required)")

In [ ]:
# Cell 4 — Configuration constants
CLIP_MODEL_NAME = "google/siglip-base-patch16-224"
VECTOR_DIM = 768
COLLECTION_NAME = "product_catalog"
TOP_K = 5
DATA_CSV_PATH = "/content/products.csv"
SUPPORTED_IMAGE_FORMATS = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

print(f"Config loaded ✓")
print(f"  Model: {CLIP_MODEL_NAME}")
print(f"  Vector dim: {VECTOR_DIM}")
print(f"  Collection: {COLLECTION_NAME}")
print(f"  Top-K: {TOP_K}")
print(f"  Data path: {DATA_CSV_PATH}")

In [ ]:
df = pd.read_csv('/content/products.csv')
df.head()

In [ ]:
# Cell 5 — DataIngestionAgent

class DataIngestionAgent:
    """Loads, validates, and returns product data from CSV."""

    REQUIRED_COLUMNS = {"product_id", "image_path", "name", "text_description"}
    OPTIONAL_DEFAULTS = {"category": "", "price": 0.0, "brand": ""}

    def __init__(self, csv_path: str):
        self.csv_path = csv_path

    def load(self) -> list[dict]:
        """Load and validate products.csv, return list of product dicts."""
        if not os.path.exists(self.csv_path):
            raise FileNotFoundError(f"CSV not found: {self.csv_path}")

        df = pd.read_csv(self.csv_path)

        # Check required columns
        missing = self.REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"CSV missing required columns: {missing}")

        # Fill optional columns with defaults
        for col, default in self.OPTIONAL_DEFAULTS.items():
            if col not in df.columns:
                df[col] = default

        # Drop rows with missing required fields
        before_count = len(df)
        df = df.dropna(subset=list(self.REQUIRED_COLUMNS))
        dropped = before_count - len(df)
        if dropped > 0:
            warnings.warn(f"Dropped {dropped} rows with missing required fields")

        products = df.to_dict(orient="records")
        print(f"DataIngestionAgent: Loaded {len(products)} products from {self.csv_path}")
        return products

print("DataIngestionAgent defined ✓")

In [ ]:
# Cell 6A — CLIPEncoderAgent (URL mode)
# ⚡ USE THIS CELL when your CSV has image URLs (http/https) in the image_path column
# ⚠️ Skip this cell if using local image files — use Cell 6B instead

class CLIPEncoderAgent:
    """Encodes images and text into the same 768-dim vector space using SigLIP.
    This version supports image URLs (downloads images from the internet)."""

    def __init__(self, model_name: str = CLIP_MODEL_NAME):
        print(f"CLIPEncoderAgent [URL mode]: Loading model '{model_name}'...")
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        print(f"CLIPEncoderAgent [URL mode]: Model loaded ✓")

    def _validate_image(self, image_path: str) -> Image.Image:
        """Download and open an image from a URL."""
        if not image_path.startswith("http://") and not image_path.startswith("https://"):
            raise ValueError(
                f"Expected a URL starting with http:// or https://, got: '{image_path}'. "
                "Use Cell 6B (local mode) for local file paths."
            )

        try:
            response = requests.get(image_path, timeout=15)
            response.raise_for_status()
            img = Image.open(io.BytesIO(response.content)).convert("RGB")
            img.verify()  # Check for corruption
            img = Image.open(io.BytesIO(response.content)).convert("RGB")  # Re-open after verify
            return img
        except requests.exceptions.RequestException as e:
            raise ValueError(f"Failed to download image from '{image_path}': {e}")
        except Exception as e:
            raise ValueError(f"Cannot read image from URL '{image_path}': {e}")

    def _l2_normalize(self, vector: np.ndarray) -> np.ndarray:
        """L2-normalize a vector for cosine similarity."""
        norm = np.linalg.norm(vector)
        if norm == 0:
            return vector
        return vector / norm

    def encode_image(self, image_path: str) -> np.ndarray:
        """Encode an image (from URL) into a 768-dim L2-normalized vector."""
        img = self._validate_image(image_path)

        with torch.no_grad():
            inputs = self.processor(images=img, return_tensors="pt")
            image_features = self.model.get_image_features(**inputs)
            vector = image_features.pooler_output.squeeze().numpy()

        return self._l2_normalize(vector)

    def encode_text(self, text: str) -> np.ndarray:
        """Encode text into a 768-dim L2-normalized vector."""
        if not text or not text.strip():
            raise ValueError("Cannot encode empty text")

        # with torch.no_grad():
        #     inputs = self.processor(text=[text], return_tensors="pt", padding=True, truncation=True)
        #     text_features = self.model.get_text_features(**inputs)
        #     vector = image_features.last_hidden_state.mean(dim=1).squeeze().numpy()
        with torch.no_grad():
          inputs = self.processor(text=[text], return_tensors="pt", padding=True, truncation=True)
          text_features = self.model.get_text_features(**inputs)
          vector = text_features.pooler_output.squeeze().numpy()

        return self._l2_normalize(vector)

print("CLIPEncoderAgent defined ✓ [URL mode — for CSVs with image URLs]")

In [ ]:
# # Cell 6B — CLIPEncoderAgent (Local file mode)
# # ⚡ USE THIS CELL when your CSV has local file paths in the image_path column
# # ⚠️ Skip this cell if using image URLs — use Cell 6A instead

# class CLIPEncoderAgent:
#     """Encodes images and text into the same 768-dim vector space using SigLIP.
#     This version supports local image file paths."""

#     def __init__(self, model_name: str = CLIP_MODEL_NAME):
#         print(f"CLIPEncoderAgent [Local mode]: Loading model '{model_name}'...")
#         self.processor = AutoProcessor.from_pretrained(model_name)
#         self.model = AutoModel.from_pretrained(model_name)
#         self.model.eval()
#         print(f"CLIPEncoderAgent [Local mode]: Model loaded ✓")

#     def _validate_image(self, image_path: str) -> Image.Image:
#         """Validate and open an image from a local file path."""
#         if not os.path.exists(image_path):
#             raise FileNotFoundError(f"Image not found: {image_path}")

#         ext = os.path.splitext(image_path)[1].lower()
#         if ext not in SUPPORTED_IMAGE_FORMATS:
#             raise ValueError(
#                 f"Unsupported image format '{ext}'. Supported: {SUPPORTED_IMAGE_FORMATS}"
#             )

#         try:
#             img = Image.open(image_path).convert("RGB")
#             img.verify()  # Check for corruption
#             img = Image.open(image_path).convert("RGB")  # Re-open after verify
#             return img
#         except Exception as e:
#             raise ValueError(f"Cannot read image '{image_path}': {e}")

#     def _l2_normalize(self, vector: np.ndarray) -> np.ndarray:
#         """L2-normalize a vector for cosine similarity."""
#         norm = np.linalg.norm(vector)
#         if norm == 0:
#             return vector
#         return vector / norm

#     def encode_image(self, image_path: str) -> np.ndarray:
#         """Encode an image (from local path) into a 768-dim L2-normalized vector."""
#         img = self._validate_image(image_path)

#         with torch.no_grad():
#             inputs = self.processor(images=img, return_tensors="pt")
#             image_features = self.model.get_image_features(**inputs)
#             vector = image_features.squeeze().numpy()

#         return self._l2_normalize(vector)

#     def encode_text(self, text: str) -> np.ndarray:
#         """Encode text into a 768-dim L2-normalized vector."""
#         if not text or not text.strip():
#             raise ValueError("Cannot encode empty text")

#         with torch.no_grad():
#             inputs = self.processor(text=[text], return_tensors="pt", padding=True, truncation=True)
#             text_features = self.model.get_text_features(**inputs)
#             vector = text_features.squeeze().numpy()

#         return self._l2_normalize(vector)

# print("CLIPEncoderAgent defined ✓ [Local mode — for CSVs with local file paths]")

In [ ]:
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

class VectorStoreAgent:
    """Manages Qdrant in-memory vector database for product vectors."""

    def __init__(self, collection_name: str = COLLECTION_NAME, vector_dim: int = VECTOR_DIM):
        # For persistence, change ":memory:" to QdrantClient(path="./qdrant_data")
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name

        # Create collection
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=vector_dim,
                distance=Distance.COSINE,
            ),
        )
        print(f"VectorStoreAgent: Collection '{self.collection_name}' created (dim={vector_dim}, COSINE)")

    def upsert(self, point_id: int, vector: np.ndarray, payload: dict):
        """Insert or update a single product vector with metadata."""
        self.client.upsert(
            collection_name=self.collection_name,
            points=[
                PointStruct(
                    id=point_id,
                    vector=vector.tolist(),
                    payload=payload,
                )
            ],
        )

    def search(self, query_vector: np.ndarray, top_k: int = TOP_K) -> list:
        """Search for the most similar products using available client methods."""
        # Try modern query_points first, then fallback to search
        if hasattr(self.client, "query_points"):
            response = self.client.query_points(
                collection_name=self.collection_name,
                query=query_vector.tolist(),
                limit=top_k,
            )
            return response.points
        else:
            return self.client.search(
                collection_name=self.collection_name,
                query_vector=query_vector.tolist(),
                limit=top_k,
            )

    def count(self) -> int:
        """Return the number of indexed points."""
        return self.client.count(collection_name=self.collection_name).count

print("VectorStoreAgent updated with robust search logic ✓")

In [ ]:
# Cell 8 — ProductSearchAgent

class ProductSearchAgent:
    """Orchestrates the full query pipeline: encode → search → return results."""

    def __init__(self, clip_encoder: CLIPEncoderAgent, vector_store: VectorStoreAgent):
        self.clip_encoder = clip_encoder
        self.vector_store = vector_store

    def _combine_vectors(self, *vectors: np.ndarray) -> np.ndarray:
        """Average multiple vectors and L2-normalize the result."""
        combined = np.mean(vectors, axis=0)
        norm = np.linalg.norm(combined)
        if norm > 0:
            combined = combined / norm
        return combined

    def search(
        self,
        query_image_path: str = None,
        query_text: str = None,
        top_k: int = TOP_K,
    ) -> list[dict]:
        """
        Search the product catalog.

        Args:
            query_image_path: Path to query image (optional)
            query_text: Text query (optional)
            top_k: Number of results to return

        Returns:
            List of dicts with rank, score, and product metadata
        """
        if query_image_path is None and (query_text is None or not query_text.strip()):
            raise ValueError("Must provide at least one of: query_image_path, query_text")

        vectors = []

        # Encode image if provided
        if query_image_path is not None:
            image_vector = self.clip_encoder.encode_image(query_image_path)
            vectors.append(image_vector)

        # Encode text if provided
        if query_text is not None and query_text.strip():
            text_vector = self.clip_encoder.encode_text(query_text)
            vectors.append(text_vector)

        # Combine vectors (average + normalize) or use single vector
        if len(vectors) == 1:
            query_vector = vectors[0]
        else:
            query_vector = self._combine_vectors(*vectors)

        # Search Qdrant
        scored_points = self.vector_store.search(query_vector, top_k=top_k)

        # Format results
        results = []
        for rank, point in enumerate(scored_points, start=1):
            result = {
                "rank": rank,
                "score": round(point.score, 4),
                **point.payload,
            }
            results.append(result)

        return results

print("ProductSearchAgent defined ✓")

In [ ]:
# Cell 9 — ResponseAgent

class ResponseAgent:
    """Formats and displays search results."""

    def format_results(self, results: list[dict], query_text: str = None) -> str:
        """Format search results into a readable string."""
        lines = []
        lines.append("=" * 60)
        if query_text:
            lines.append(f"Search Query: \"{query_text}\"")
        lines.append(f"Top {len(results)} Results:")
        lines.append("=" * 60)

        for r in results:
            lines.append(f"\n  #{r['rank']}  Score: {r['score']}")
            lines.append(f"       Name: {r.get('name', 'N/A')}")
            lines.append(f"         ID: {r.get('product_id', 'N/A')}")
            if r.get('brand'):
                lines.append(f"      Brand: {r['brand']}")
            if r.get('category'):
                lines.append(f"   Category: {r['category']}")
            if r.get('price'):
                lines.append(f"      Price: ${r['price']:.2f}")
            lines.append(f"       Desc: {r.get('text_description', 'N/A')[:80]}...")

        lines.append("\n" + "=" * 60)
        return "\n".join(lines)

    def display(self, results: list[dict], query_text: str = None):
        """Print formatted results to stdout."""
        output = self.format_results(results, query_text)
        print(output)

print("ResponseAgent defined ✓")

In [ ]:
# Cell 10 — Initialize all agents

# Data ingestion
ingestion_agent = DataIngestionAgent(csv_path=DATA_CSV_PATH)

# CLIP encoder
clip_encoder = CLIPEncoderAgent(model_name=CLIP_MODEL_NAME)

# Vector store (Updated with robust search logic)
vector_store = VectorStoreAgent(collection_name=COLLECTION_NAME, vector_dim=VECTOR_DIM)

# Search orchestrator
search_agent = ProductSearchAgent(clip_encoder=clip_encoder, vector_store=vector_store)

# Response formatter
response_agent = ResponseAgent()

print("\n✓ All agents re-initialized successfully")

## Section 4 — Ingest Phase

The ingest loop processes every product in the CSV:
1. Encode each product's **image** → 768-dim vector
2. Encode each product's **text description** → 768-dim vector
3. **Average** both vectors + L2-normalize → combined product vector
4. **Upsert** into Qdrant with product metadata as payload

This is **entirely local and free** — no API calls, no rate limits.

In [ ]:
# Cell 12 — Ingest loop

products = ingestion_agent.load()
failed_products = []
successful_count = 0

for idx, product in enumerate(products):
    product_id = product["product_id"]
    image_path = product["image_path"]
    text_desc = product["text_description"]

    try:
        image_vector = None
        text_vector = None

        # Encode image
        try:
            image_vector = clip_encoder.encode_image(image_path)
        except Exception as e:
            pass

        # Encode text
        try:
            text_vector = clip_encoder.encode_text(text_desc)
        except Exception as e:
            pass

        # Combine vectors
        if image_vector is not None and text_vector is not None:
            combined = np.mean([image_vector, text_vector], axis=0)
            norm = np.linalg.norm(combined)
            if norm > 0:
                combined = combined / norm
        elif image_vector is not None:
            combined = image_vector
        elif text_vector is not None:
            combined = text_vector
        else:
            raise ValueError("Both image and text encoding failed")

        payload = {
            "product_id": product_id,
            "name": product["name"],
            "text_description": text_desc,
            "image_path": image_path,
            "category": product.get("category", ""),
            "price": float(product.get("price", 0.0)),
            "brand": product.get("brand", ""),
        }

        vector_store.upsert(point_id=idx, vector=combined, payload=payload)
        successful_count += 1

    except Exception as e:
        failed_products.append({"product_id": product_id, "error": str(e)})

print(f"\nIngest complete: {successful_count}/{len(products)} products indexed")

## Section 5 — Query Phase

Now we can search the catalog! Three query modes:
1. **Image + Text** — provides the most balanced results
2. **Image only** — finds visually similar products
3. **Text only** — finds semantically similar products

In [ ]:
# Cell 14 — Example: Image + Text query
QUERY_IMAGE_PATH = "https://cb2.scene7.com/is/image/CB2/TauBrnAshWdCredenza70inSHF24/$web_plp_card$/240922084338/TauBrnAshWdCredenza70inSHF24.jpg"
QUERY_TEXT = "wood"

print(f"Query image: {QUERY_IMAGE_PATH}")
print(f"Query text: \"{QUERY_TEXT}\"\n")

results = search_agent.search(query_image_path=QUERY_IMAGE_PATH, query_text=QUERY_TEXT)
response_agent.display(results, QUERY_TEXT)

In [ ]:
# Cell 15 — Example: Text-only query
TEXT_ONLY_QUERY = "walnut"

print(f"Query text: \"{TEXT_ONLY_QUERY}\" (no image)\n")

results = search_agent.search(query_image_path=None, query_text=TEXT_ONLY_QUERY)
response_agent.display(results, TEXT_ONLY_QUERY)

## Section 6 — Verification

Automated checks to confirm the system is working correctly.

In [ ]:
# Cell 17 — Verify indexed count
expected_count = len(products) - len(failed_products)
actual_count = vector_store.count()

assert actual_count == expected_count, (
    f"Count mismatch! Expected {expected_count}, got {actual_count}"
)
print(f"✓ Vector store count: {actual_count} (matches expected {expected_count})")

In [ ]:
# Cell 18 — Inspect a stored point
point = vector_store.client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[0],
    with_vectors=True,
)

if point:
    p = point[0]
    print(f"Point ID: {p.id}")
    print(f"Payload keys: {list(p.payload.keys())}")
    print(f"Vector length: {len(p.vector)}")
    print(f"Sample payload: {p.payload['name']} — ${p.payload.get('price', 'N/A')}")
    assert len(p.vector) == VECTOR_DIM, f"Vector dim mismatch! Expected {VECTOR_DIM}, got {len(p.vector)}"
    print(f"✓ Vector dimension verified: {VECTOR_DIM}")
else:
    print("✗ No point found at id=0")

In [ ]:
# Cell 19 — Self-consistency test
# Search product[0] using its own image + description → it should rank #1
test_product = products[0]
print(f"Self-consistency test: searching for '{test_product['name']}'")
print(f"  Image: {test_product['image_path']}")
print(f"  Text: {test_product['text_description'][:60]}...\n")

test_results = search_agent.search(
    query_image_path=test_product["image_path"],
    query_text=test_product["text_description"],
)

top_result = test_results[0]
assert top_result["product_id"] == test_product["product_id"], (
    f"Self-consistency FAILED! Expected '{test_product['product_id']}' at rank #1, "
    f"got '{top_result['product_id']}' (score: {top_result['score']})"
)

print(f"✓ Self-consistency passed: '{top_result['name']}' ranked #1 with score {top_result['score']}")
print(f"\nFull results:")
response_agent.display(test_results, test_product["text_description"][:60])